# AssureX Claim Engine — Data Preprocessing & Python Model Training

**Project:** AssureX Claim Engine  
**SRS Reference:** Sections 1.2 Steps 3–5 (Preprocessing, Dataset Creation, Python Model Training)

This notebook covers:
1. Data loading and cleaning
2. Feature engineering (product age, remaining warranty, etc.)
3. Categorical encoding and numerical normalization
4. Training and comparing 3 ML algorithms
5. Best model selection and evaluation
6. Confusion matrix and per-class metrics
7. Feature importance analysis
8. Model export (joblib .pkl)


## 1. Setup & Imports

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix,
    f1_score, precision_score, recall_score, ConfusionMatrixDisplay
)
from sklearn.model_selection import cross_val_score
import joblib
from pathlib import Path

BASE_DIR = Path('..')
PROC_DIR = BASE_DIR / 'data' / 'processed'
MODEL_DIR = BASE_DIR / 'model' / 'python_model'

print("All imports successful.")


## 2. Load & Inspect Data

In [ ]:
train_df = pd.read_csv(PROC_DIR / 'train.csv')
val_df   = pd.read_csv(PROC_DIR / 'validation.csv')
test_df  = pd.read_csv(PROC_DIR / 'test.csv')

print(f"Train: {train_df.shape}  |  Validation: {val_df.shape}  |  Test: {test_df.shape}")
train_df.head(3)


## 3. Feature Engineering & Preprocessing

In [ ]:
# Feature groups (must match src/preprocessing/claim_preprocessor.py)
CATEGORICAL_FEATURES = [
    'damage_type', 'warranty_type', 'product_category',
    'has_purchase_receipt', 'has_warranty_card', 'has_product_image',
    'has_fault_evidence', 'has_repair_report', 'serial_number_match',
    'had_unauthorized_repair', 'is_duplicate_flag',
]
NUMERICAL_FEATURES = [
    'product_age_months', 'remaining_warranty_days', 'days_since_fault',
    'claim_submission_delay_days', 'repair_count', 'missing_doc_count', 'purchase_price',
]
TARGET = 'label'

# Separate features from target
FEATURE_COLS = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

def prepare_split(df):
    X = df[FEATURE_COLS].copy()
    y = df[TARGET].copy()
    # Fill any missing values
    X[CATEGORICAL_FEATURES] = X[CATEGORICAL_FEATURES].fillna('unknown')
    X[NUMERICAL_FEATURES]   = X[NUMERICAL_FEATURES].fillna(0)
    return X, y

X_train, y_train = prepare_split(train_df)
X_val,   y_val   = prepare_split(val_df)
X_test,  y_test  = prepare_split(test_df)

print(f"X_train: {X_train.shape}  |  Features: {FEATURE_COLS}")
print(f"Classes: {sorted(y_train.unique())}")


In [ ]:
# Build preprocessing pipeline
# Categorical: OneHotEncoder for string cols, passthrough for binary int cols
str_cat = ['damage_type', 'warranty_type', 'product_category']
bin_cat  = [c for c in CATEGORICAL_FEATURES if c not in str_cat]

preprocessor = ColumnTransformer(transformers=[
    ('str_cat',  OneHotEncoder(handle_unknown='ignore', sparse_output=False), str_cat),
    ('bin_pass', 'passthrough', bin_cat),
    ('num_scale', StandardScaler(), NUMERICAL_FEATURES),
], remainder='drop')

print("Preprocessor built.")
print(f"  String categoricals : {str_cat}")
print(f"  Binary passthrough  : {bin_cat}")
print(f"  Numerical (scaled)  : {NUMERICAL_FEATURES}")


## 4. Model Training & Comparison

In [ ]:
# Label encoder
le = LabelEncoder()
le.fit(['invalid_claim', 'manual_review', 'valid_claim'])
print("Classes:", le.classes_)


In [ ]:
# Define 3 models to compare (SRS requires at least 3)
models = {
    "Random Forest": Pipeline([
        ('pre', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)),
    ]),
    "Logistic Regression": Pipeline([
        ('pre', preprocessor),
        ('clf', LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs', random_state=42)),
    ]),
    "Gradient Boosting": Pipeline([
        ('pre', preprocessor),
        ('clf', GradientBoostingClassifier(n_estimators=100, learning_rate=0.1,
                                            max_depth=4, random_state=42)),
    ]),
}

results = {}
for name, pipeline in models.items():
    print(f"Training: {name} ...", end=' ', flush=True)
    pipeline.fit(X_train, y_train)
    val_pred  = pipeline.predict(X_val)
    test_pred = pipeline.predict(X_test)
    val_acc   = accuracy_score(y_val,  val_pred)  * 100
    test_acc  = accuracy_score(y_test, test_pred) * 100
    val_f1    = f1_score(y_val, val_pred, average='weighted', zero_division=0) * 100
    cv        = cross_val_score(pipeline, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = {
        'pipeline':   pipeline,
        'val_acc':    round(val_acc, 2),
        'test_acc':   round(test_acc, 2),
        'val_f1':     round(val_f1, 2),
        'cv_mean':    round(cv.mean() * 100, 2),
        'cv_std':     round(cv.std()  * 100, 2),
        'test_pred':  test_pred,
    }
    print(f"val={val_acc:.2f}%  test={test_acc:.2f}%  cv={cv.mean()*100:.2f}%")

print("\nAll models trained.")


In [ ]:
# Print comparison table
print("\n" + "=" * 60)
print(f"{'Model':<25} {'Val Acc':>8} {'Test Acc':>9} {'CV Acc':>8} {'Val F1':>7}")
print("-" * 60)
for name, r in results.items():
    print(f"{name:<25} {r['val_acc']:>7.2f}%  {r['test_acc']:>8.2f}%  "
          f"{r['cv_mean']:>6.2f}%  {r['val_f1']:>6.2f}%")
print("=" * 60)

best_name = max(results, key=lambda n: results[n]['test_acc'])
print(f"\nBest model: {best_name}  (Test Acc = {results[best_name]['test_acc']:.2f}%)")
print(f"SRS 85% accuracy target: {'MET ✓' if results[best_name]['test_acc'] >= 85 else 'NOT MET ✗'}")


## 5. Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
DISPLAY_LABELS = ['Invalid', 'Manual Review', 'Valid']
CLASSES_ORDERED = ['invalid_claim', 'manual_review', 'valid_claim']

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['test_pred'], labels=CLASSES_ORDERED)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=DISPLAY_LABELS)
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\nTest Acc = {r["test_acc"]:.2f}%', fontweight='bold')

plt.suptitle('Confusion Matrices — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/model_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/model_confusion_matrices.png")


## 6. Classification Report — Best Model

In [ ]:
best_pipeline = results[best_name]['pipeline']
test_pred     = results[best_name]['test_pred']

print(f"Classification Report: {best_name}")
print("=" * 55)
print(classification_report(y_test, test_pred, target_names=DISPLAY_LABELS, zero_division=0))


## 7. Feature Importance (Best Model)

In [ ]:
clf = best_pipeline.named_steps['clf']

if hasattr(clf, 'feature_importances_'):
    pre  = best_pipeline.named_steps['pre']
    # Get feature names after transformation
    ohe_names = list(pre.named_transformers_['str_cat'].get_feature_names_out(str_cat))
    all_names = ohe_names + bin_cat + NUMERICAL_FEATURES

    importances = clf.feature_importances_
    indices     = np.argsort(importances)[::-1][:20]   # top 20

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(range(len(indices)),
           importances[indices],
           color='#2563EB', edgecolor='black', alpha=0.85)
    ax.set_xticks(range(len(indices)))
    ax.set_xticklabels(
        [all_names[i] if i < len(all_names) else f'feat_{i}' for i in indices],
        rotation=45, ha='right', fontsize=9
    )
    ax.set_title(f'Feature Importance — {best_name} (Top 20)', fontsize=13, fontweight='bold')
    ax.set_ylabel('Importance Score')
    plt.tight_layout()
    plt.savefig('../reports/model_feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved → reports/model_feature_importance.png")
else:
    print(f"{best_name} does not support feature_importances_ directly.")


## 8. Cross-Validation Results

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
names = list(results.keys())
means = [results[n]['cv_mean'] for n in names]
stds  = [results[n]['cv_std']  for n in names]

bars = ax.bar(names, means, yerr=stds, capsize=8,
              color=['#2563EB', '#F59E0B', '#16A34A'], edgecolor='black', alpha=0.85)
ax.axhline(85, color='red', linestyle='--', linewidth=1.5, label='SRS Target (85%)')
ax.set_title('5-Fold Cross-Validation Accuracy', fontsize=13, fontweight='bold')
ax.set_ylabel('Accuracy (%)')
ax.set_ylim(80, 102)
ax.legend()

for bar, mean, std in zip(bars, means, stds):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + std + 0.3,
            f'{mean:.2f}%', ha='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.savefig('../reports/model_cv_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved → reports/model_cv_results.png")


## 9. Save Best Model

In [ ]:
# Load already-trained model (saved by src/ml/train_model.py)
import json

model_path = MODEL_DIR / 'assurex_model.pkl'
le_path    = MODEL_DIR / 'label_encoder.pkl'
pre_path   = MODEL_DIR / 'preprocessor.pkl'

if model_path.exists():
    saved_model = joblib.load(model_path)
    print(f"✓ Saved model loaded from {model_path}")
    print(f"  Type: {type(saved_model)}")
else:
    # Save the best pipeline from this notebook
    joblib.dump(best_pipeline, model_path)
    joblib.dump(le,            le_path)
    joblib.dump(preprocessor,  pre_path)
    print(f"✓ Best model saved → {model_path}")

# Load training report
report_path = MODEL_DIR / 'training_report.txt'
if report_path.exists():
    print("\n--- Training Report ---")
    print(report_path.read_text(encoding='utf-8'))


## 10. Summary

In [ ]:
print("=" * 60)
print("AssureX — Python Model Training Summary")
print("=" * 60)
print(f"  Dataset split  : 1050 train / 225 val / 225 test")
print(f"  Features       : {len(FEATURE_COLS)}")
print(f"  Models compared: {len(results)}")
print()
for name, r in results.items():
    mark = ' ◀ BEST' if name == best_name else ''
    print(f"  {name:<25} Test={r['test_acc']:.2f}%  CV={r['cv_mean']:.2f}%{mark}")
print()
print(f"  Best model     : {best_name}")
print(f"  Best test acc  : {results[best_name]['test_acc']:.2f}%")
print(f"  SRS 85% target : MET ✓" if results[best_name]['test_acc'] >= 85 else "  SRS 85% target : NOT MET ✗")
print("=" * 60)
